# Bonus — Advanced SDP-Meta Concepts

This notebook covers advanced SDP-Meta features beyond the core labs. Each section describes a capability you can configure through your onboarding files to extend your pipelines. These topics can be explored independently after completing Labs 1–3.

## Snapshots

Snapshot ingestion handles source systems that export periodic full snapshots (e.g., a nightly Parquet or Delta dump of an entire table) rather than streaming change events. SDP-Meta automatically diffs successive snapshots to identify inserts, updates, and deletes, then applies them as SCD Type 1 or Type 2 changes.

### Source Configuration

Set the snapshot format under `source_details` in your onboarding JSON. Note the use of `snapshot_format` instead of `source_format`:

```json
"source_details": {
  "snapshot_format": "parquet",
  "source_path_dev": "s3://my-bucket/snapshots/customers/",
  "source_path_prod": "s3://my-prod-bucket/snapshots/customers/"
}
```

Supported snapshot formats include `parquet`, `delta`, and `json`.

### Bronze Snapshot Apply

Add `bronze_apply_changes_from_snapshot` to your onboarding entry to enable snapshot-diff on the bronze table:

```json
"bronze_apply_changes_from_snapshot": {
  "keys": ["customer_id"],
  "scd_type": "1"
}
```

| Field | Description |
| --- | --- |
| `keys` | Primary key column(s) used to match records across snapshots |
| `scd_type` | `"1"` overwrites the existing row (no history). `"2"` retains full row history with `__START_AT` / `__END_AT` columns |

### Silver Snapshot Apply

Configure `silver_apply_changes_from_snapshot` for the same snapshot-apply semantics on the silver table:

```json
"silver_apply_changes_from_snapshot": {
  "keys": ["customer_id"],
  "scd_type": "2"
}
```

You can use different SCD types at each layer — for example, Type 1 on bronze (always current state) and Type 2 on silver (full history with temporal columns).

### When to Use Snapshots vs CDC

| Approach | Use When |
| --- | --- |
| **Snapshot** (`*_apply_changes_from_snapshot`) | Source provides periodic full exports; no change events available |
| **CDC** (`*_cdc_apply_changes`) | Source emits a stream of row-level insert / update / delete events |

> **Note:** Snapshot and CDC fields are mutually exclusive on the same onboarding entry. If your source provides both, choose the approach that best matches your latency and storage requirements.


## Notebook Hooks

SDP-Meta’s `DataflowPipeline.invoke_dlt_pipeline` method accepts optional callback parameters that let you inject custom logic into the pipeline without modifying the framework itself. These hooks are passed in the **runner notebook** (`notebooks/init_sdp_meta_pipeline`).

### Custom Transform Functions

Apply a custom transformation to every DataFrame in the bronze or silver layer:

| Parameter | Description |
| --- | --- |
| `bronze_custom_transform_func` | A function applied to each bronze DataFrame after ingestion |
| `silver_custom_transform_func` | A function applied to each silver DataFrame after transformation |

A custom transform function receives the DataFrame and the dataflowspec row as arguments and returns a modified DataFrame:

```python
def my_bronze_transform(df, dataflow_spec):
    """Add a custom audit column to every bronze table."""
    from pyspark.sql import functions as F
    return df.withColumn("_ingested_at", F.current_timestamp())
```

Then pass it when invoking the pipeline in your runner notebook:

```python
from databricks.labs.sdp_meta.dataflow_pipeline import DataflowPipeline

DataflowPipeline.invoke_dlt_pipeline(
    spark,
    layer,
    bronze_custom_transform_func=my_bronze_transform
)
```

### Snapshot Version Functions

For snapshot-based pipelines, you can supply custom functions to control which snapshot version to process next:

| Parameter | Description |
| --- | --- |
| `bronze_next_snapshot_and_version` | Returns the next snapshot DataFrame and its version identifier for bronze |
| `silver_next_snapshot_and_version` | Returns the next snapshot DataFrame and its version identifier for silver |

These are useful when your snapshot storage doesn’t follow a simple path-based versioning pattern and you need custom logic to locate and load the next unprocessed snapshot.

### Example: Applying a Hook in the Runner Notebook

To use hooks in this lab, you would modify `notebooks/init_sdp_meta_pipeline` to pass your functions:

```python
from databricks.labs.sdp_meta.dataflow_pipeline import DataflowPipeline

def add_audit_columns(df, spec):
    from pyspark.sql import functions as F
    return (
        df.withColumn("_pipeline_run_id", F.lit(spark.conf.get("pipeline_run_id", "unknown")))
          .withColumn("_processed_at", F.current_timestamp())
    )

DataflowPipeline.invoke_dlt_pipeline(
    spark,
    layer,
    bronze_custom_transform_func=add_audit_columns,
    silver_custom_transform_func=add_audit_columns
)
```

> **Tip:** Custom transform functions are applied to every table in that layer. Use the `dataflow_spec` argument to conditionally apply logic to specific tables based on `data_flow_id` or other metadata fields.


## Row Filters

Row filters let you restrict which rows flow from bronze to silver by applying SQL `WHERE` clause conditions in your configuration files. This is useful for excluding test records, filtering by date range, or scoping tables to a specific subset of the data.

### Silver Transformations

Row filters are defined in the silver transformations configuration file (referenced by the `silver_transformation_json` field in your onboarding entry). Each transformation entry can include a `where_clause`:

```json
[
  {
    "target_table": "host_status_updates_csv_silver",
    "target_partition_cols": ["region"],
    "select_exp": [
      "system_id",
      "status",
      "region",
      "hostname",
      "event_time"
    ],
    "where_clause": "status != 'TERMINATED' AND region IN ('us-east', 'us-west')"
  }
]
```

| Field | Description |
| --- | --- |
| `target_table` | The silver table this transformation applies to |
| `target_partition_cols` | Columns to partition the silver table by |
| `select_exp` | SQL expressions selecting and transforming columns from the bronze table |
| `where_clause` | SQL predicate filtering rows (applied as a `WHERE` condition) |

### Referencing the Transformations File

Point to your transformations JSON in the onboarding entry:

```json
"silver_transformation_json": "/Volumes/<catalog>/<schema>/<volume>/conf/silver_transformations.json"
```

The file is staged to the UC Volume by the `stage_conf` task during onboarding, so use the volume path (not the workspace path).

### Use Cases

* **Exclude test data**: `"where_clause": "environment != 'test'"`
* **Regional scoping**: `"where_clause": "region = 'us-east'"`
* **Date windowing**: `"where_clause": "event_time >= '2025-01-01'"`
* **Status filtering**: `"where_clause": "status NOT IN ('TERMINATED', 'DECOMMISSIONED')"`

### Combining with Column Selection

The `select_exp` array also acts as a column filter — only the listed expressions are passed to the silver table. You can rename columns, cast types, or compute derived values:

```json
"select_exp": [
  "system_id",
  "UPPER(status) AS status",
  "region",
  "hostname",
  "CAST(event_time AS TIMESTAMP) AS event_time"
]
```

> **Tip:** Row filters can be combined with the split-pipeline technique from Lab 3 — use `silver_append_flows` with per-flow filter conditions to route different subsets of data to different silver tables.

## Pipeline Modes

SDP-Meta supports two pipeline modes that control how bronze and silver layers are organized into Lakeflow Declarative Pipelines. The mode is set via the `pipeline_mode` bundle variable in `resources/variables.yml`.

### Combined Mode (Default)

```yaml
pipeline_mode: combined
```

A **single** Lakeflow Declarative Pipeline materializes both bronze and silver tables in one DAG. SDP-Meta reads both `bronze.dataflowspecTable` and `silver.dataflowspecTable` from the pipeline configuration and builds the entire graph together.

**Advantages:**
* Simpler to manage — one pipeline, one job, one monitoring view
* Lower overhead for small to medium workloads
* Silver tables automatically refresh when bronze tables update within the same pipeline run

This is the mode used throughout Labs 1–3 in this repository.

### Split Mode

```yaml
pipeline_mode: split
```

**Two separate** Lakeflow Declarative Pipelines are created: one for bronze and one for silver. In the bundle’s job definition, the silver pipeline task depends on the bronze pipeline task, so they run in sequence.

**Advantages:**
* Independent scaling — bronze and silver can use different compute configurations
* Isolation — a silver failure doesn’t block bronze ingestion
* Flexibility — different refresh cadences (e.g., bronze streams continuously, silver refreshes hourly)

### When to Use Split Mode

* Your bronze layer ingests high-volume streaming data that needs dedicated compute
* Bronze and silver have different SLA or refresh schedules
* You want to run bronze continuously but silver on a trigger
* Different teams own different layers and need independent deployability

### How to Configure

Set the variable in your `variable-overrides.json` or `databricks.yml` target:

```json
"pipeline_mode": "split"
```

Or override at deploy time:

```bash
databricks bundle deploy --target dev --var pipeline_mode=split
```

The bundle’s resource templates in `resources/sdp_meta_pipelines.yml` respond to this variable and create one or two pipelines accordingly.

> **Note:** When the `layer` variable is set to only `bronze` or only `silver` (instead of `bronze_silver`), the pipeline mode variable is ignored — only the requested layer’s pipeline is created.

## Building and Deploying Custom SDP-META Wheel Files

By default, this lab installs SDP-Meta from PyPI using the `sdp_meta_dependency` variable (e.g., `databricks-labs-sdp-meta==0.1.0`). For development or customization, you can build a wheel from source and deploy it to a UC Volume.

### Step 1: Clone and Build the Wheel

```bash
# Clone the sdp-meta source repository
git clone https://github.com/databrickslabs/dlt-meta.git
cd dlt-meta

# Build the wheel
pip install build
python -m build --wheel
```

The built wheel will be in `dist/`, e.g., `dist/databricks_labs_sdp_meta-0.1.0-py3-none-any.whl`.

### Step 2: Upload to a UC Volume

Upload the wheel to your UC Volume so that serverless pipelines can access it:

```bash
# Using the Databricks CLI
databricks fs cp dist/databricks_labs_sdp_meta-0.1.0-py3-none-any.whl \
  dbfs:/Volumes/<catalog>/<schema>/<volume>/wheel/databricks_labs_sdp_meta-0.1.0-py3-none-any.whl \
  --overwrite
```

Or use the bundle’s built-in script, which copies the wheel from the `wheels/` directory in this repo:

```bash
databricks bundle run upload_sdp_meta_wheel --target dev
```

### Step 3: Update Bundle Variables

Switch the bundle from PyPI to volume-based installation by updating two variables in your `variable-overrides.json`:

```json
{
  "wheel_source": "volume_path",
  "sdp_meta_dependency": "/Volumes/<catalog>/<schema>/<volume>/wheel/databricks_labs_sdp_meta-0.1.0-py3-none-any.whl"
}
```

| Variable | PyPI (Default) | Volume Wheel |
| --- | --- | --- |
| `wheel_source` | `pypi` | `volume_path` |
| `sdp_meta_dependency` | `databricks-labs-sdp-meta==0.1.0` | `/Volumes/<catalog>/<schema>/<volume>/wheel/...whl` |

### Step 4: Redeploy

```bash
databricks bundle deploy --target dev
```

The runner notebook (`notebooks/init_sdp_meta_pipeline`) reads `sdp_meta_dependency` from the pipeline configuration and runs `%pip install $sdp_meta_dependency`, so the switch between PyPI and volume wheel is seamless — no other code changes are needed.

### Pre-Built Wheel

This repository already includes a pre-built wheel at `wheels/databricks_labs_sdp_meta-0.1.0-py3-none-any.whl`. The `upload_sdp_meta_wheel` bundle script uploads it for you — useful if you just want to pin a known version without building from source.

> **Tip:** When developing custom SDP-Meta features, iterate locally by building the wheel, uploading via the bundle script, and redeploying. The pipeline will pick up your changes on the next run.